# Semana 13: Interpretabilidad de Modelos
## Notebook de Ejercicios (NB2) – Explicación de un Modelo de Crédito

**Propósito:** Aplicar técnicas de interpretabilidad (SHAP) a un modelo de riesgo crediticio, generando explicaciones globales y locales, y redactar un informe para un cliente simulando la explicación de por qué se le negó el crédito.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Entrenar un modelo XGBoost para clasificación de riesgo crediticio.
- Generar explicaciones globales usando SHAP (summary plot, dependence plots).
- Generar explicaciones locales para solicitudes aprobadas y rechazadas.
- Redactar un informe comprensible para un cliente simulado.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# XGBoost
import xgboost as xgb

# SHAP
import shap

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: German Credit

El dataset **German Credit** contiene información sobre solicitantes de crédito y una clasificación de riesgo (bueno o malo). Está disponible en el repositorio UCI.

In [ ]:
# URL del dataset (UCI)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'

# Nombres de las columnas según la documentación
column_names = [
    'status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status', 'other_debtors',
    'residence_since', 'property', 'age', 'other_installment', 'housing',
    'existing_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'target'
]

try:
    df = pd.read_csv(url, sep=' ', header=None, names=column_names)
    print("Dataset cargado correctamente.")
except:
    print("No se pudo cargar desde UCI. Usando datos locales...")
    # Datos sintéticos similares
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'status': np.random.randint(1, 5, n),
        'duration': np.random.randint(4, 72, n),
        'credit_history': np.random.randint(0, 5, n),
        'purpose': np.random.randint(0, 10, n),
        'credit_amount': np.random.randint(250, 20000, n),
        'savings': np.random.randint(1, 5, n),
        'employment': np.random.randint(1, 5, n),
        'installment_rate': np.random.randint(1, 5, n),
        'personal_status': np.random.randint(1, 4, n),
        'other_debtors': np.random.randint(1, 4, n),
        'residence_since': np.random.randint(1, 4, n),
        'property': np.random.randint(1, 5, n),
        'age': np.random.randint(19, 75, n),
        'other_installment': np.random.randint(1, 4, n),
        'housing': np.random.randint(1, 4, n),
        'existing_credits': np.random.randint(1, 4, n),
        'job': np.random.randint(1, 4, n),
        'people_liable': np.random.randint(1, 3, n),
        'telephone': np.random.randint(1, 3, n),
        'foreign_worker': np.random.randint(1, 3, n),
        'target': np.random.choice([1, 2], n, p=[0.7, 0.3])
    })

# Convertimos target: 1 = bueno, 2 = malo -> 0 = bueno, 1 = malo
df['target'] = df['target'].map({1: 0, 2: 1})

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información general
df.info()

In [ ]:
# Distribución de la variable objetivo
print("Distribución de target:")
print(df['target'].value_counts())
print(f"\nPorcentaje de malos pagadores: {df['target'].mean()*100:.2f}%")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target')
plt.title('Distribución de Riesgo Crediticio')
plt.xticks([0, 1], ['Bueno (0)', 'Malo (1)'])
plt.show()

---
## 2. Preprocesamiento de Datos

El dataset contiene variables numéricas y categóricas. Aplicamos one-hot encoding a las categóricas.

In [ ]:
# Separamos características y objetivo
X = df.drop('target', axis=1)
y = df['target']

# Identificamos columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=[object]).columns.tolist()

print(f"Columnas numéricas: {numeric_cols}")
print(f"Columnas categóricas: {categorical_cols}")

# Para este dataset, todas las columnas son numéricas (códigos de categorías)
# Pero algunas representan categorías, así que las trataremos como numéricas
# XGBoost puede manejar categóricas si se codifican adecuadamente

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nTamaño entrenamiento: {X_train.shape}")
print(f"Tamaño prueba: {X_test.shape}")

---
## 3. Entrenamiento de XGBoost

Entrenamos un modelo XGBoost para clasificación de riesgo crediticio.

In [ ]:
# Parámetros del modelo
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

# Entrenamos
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Métricas de evaluación ===")
print(classification_report(y_test, y_pred, target_names=['Bueno', 'Malo']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

# Matriz de confusión
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - XGBoost')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

---
## 4. Explicaciones Globales con SHAP

Usamos TreeExplainer para calcular valores SHAP y generar visualizaciones globales.

In [ ]:
# Creamos el explainer
explainer = shap.TreeExplainer(model)

# Calculamos SHAP para el conjunto de prueba (usamos una muestra para velocidad)
X_test_sample = X_test.sample(100, random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print("Valores SHAP calculados.")

### 4.1. Summary Plot (importancia global con dirección)

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, feature_names=X.columns)
plt.show()

### 4.2. Bar Plot (importancia promedio absoluta)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_sample, feature_names=X.columns, plot_type='bar')
plt.show()

### 4.3. Dependence Plots para variables importantes

Visualizamos cómo afectan algunas variables clave al riesgo.

In [ ]:
# Dependence plot para duration (duración del crédito)
shap.dependence_plot('duration', shap_values, X_test_sample, feature_names=X.columns)
plt.show()

In [ ]:
# Dependence plot para credit_amount (monto del crédito)
shap.dependence_plot('credit_amount', shap_values, X_test_sample, feature_names=X.columns)
plt.show()

In [ ]:
# Dependence plot para age (edad)
shap.dependence_plot('age', shap_values, X_test_sample, feature_names=X.columns)
plt.show()

---
## 5. Explicaciones Locales

Seleccionamos dos instancias: una aprobada (predicción buena) y una rechazada (predicción mala) para explicar localmente.

In [ ]:
# Predecimos probabilidades para todas las instancias de test
y_proba_test = model.predict_proba(X_test)[:, 1]

# Encontramos índices de aprobados y rechazados
# Aprobados: probabilidad < 0.5 (clase 0 - bueno)
# Rechazados: probabilidad >= 0.5 (clase 1 - malo)
aprobados_idx = np.where(y_proba_test < 0.5)[0]
rechazados_idx = np.where(y_proba_test >= 0.5)[0]

# Seleccionamos un ejemplo de cada tipo
idx_aprobado = aprobados_idx[0]
idx_rechazado = rechazados_idx[0]

print(f"Ejemplo aprobado - Índice: {idx_aprobado}, Probabilidad: {y_proba_test[idx_aprobado]:.4f}")
print(f"Ejemplo rechazado - Índice: {idx_rechazado}, Probabilidad: {y_proba_test[idx_rechazado]:.4f}")

### 5.1. Waterfall Plot para el caso aprobado

In [ ]:
# Calculamos SHAP para estas instancias específicas
shap_values_aprobado = explainer.shap_values(X_test.iloc[[idx_aprobado]])

shap.waterfall_plot(shap.Explanation(
    values=shap_values_aprobado[0],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx_aprobado].values,
    feature_names=X.columns
))
plt.show()

### 5.2. Waterfall Plot para el caso rechazado

In [ ]:
shap_values_rechazado = explainer.shap_values(X_test.iloc[[idx_rechazado]])

shap.waterfall_plot(shap.Explanation(
    values=shap_values_rechazado[0],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx_rechazado].values,
    feature_names=X.columns
))
plt.show()

### 5.3. Análisis comparativo

Mostramos los valores de las variables para ambos casos.

In [ ]:
# Creamos un DataFrame comparativo
comparacion = pd.DataFrame({
    'Característica': X.columns,
    'Aprobado': X_test.iloc[idx_aprobado].values,
    'Rechazado': X_test.iloc[idx_rechazado].values,
    'SHAP Aprobado': shap_values_aprobado[0],
    'SHAP Rechazado': shap_values_rechazado[0]
})

print("Comparación de casos aprobado vs rechazado:")
comparacion.round(2)

---
## 6. Informe para el Cliente (Simulación)

Basado en el waterfall plot del caso rechazado, redactamos un informe explicando por qué se negó el crédito.

In [ ]:
# Extraemos los factores más importantes para el caso rechazado
shap_rech = pd.Series(shap_values_rechazado[0], index=X.columns)
top_factores = shap_rech.abs().sort_values(ascending=False).head(5)

print("=== INFORME PARA EL CLIENTE ===")
print("Estimado cliente,")
print("\nHemos evaluado su solicitud de crédito utilizando un modelo de machine learning.\n")
print("Lamentamos informarle que su solicitud no ha sido aprobada en esta ocasión.")
print("\nA continuación, explicamos los principales factores que influyeron en esta decisión:\n")

for factor in top_factores.index:
    valor = X_test.iloc[idx_rechazado][factor]
    contrib = shap_rech[factor]
    if contrib > 0:
        efecto = "aumentó el riesgo"
    else:
        efecto = "disminuyó el riesgo"
    print(f"- {factor}: Su valor ({valor}) {efecto} en {abs(contrib):.3f} unidades.")

print("\nEstos factores se combinan para determinar la probabilidad de incumplimiento.")
print("Le recomendamos revisar especialmente aquellos que aumentaron el riesgo.")
print("\nAtentamente,\nDepartamento de Riesgo Crediticio")

---
## 7. Análisis de Sesgos (Opcional)

Podemos verificar si variables sensibles como 'age' (edad) o 'personal_status' (estado personal/género) tienen un impacto desproporcionado.

In [ ]:
# Analizamos la distribución de SHAP para 'age'
plt.figure(figsize=(10, 5))
shap.dependence_plot('age', shap_values, X_test_sample, feature_names=X.columns)
plt.title('Dependence Plot - Edad')
plt.show()

# Verificamos si hay diferencias por grupos de edad
X_test_sample['age_group'] = pd.cut(X_test_sample['age'], bins=[0, 30, 50, 100], labels=['joven', 'medio', 'mayor'])
shap_age = pd.DataFrame({
    'age_group': X_test_sample['age_group'],
    'shap_age': shap_values[:, X.columns.get_loc('age')]
})

plt.figure(figsize=(8, 5))
sns.boxplot(data=shap_age, x='age_group', y='shap_age')
plt.title('Distribución de SHAP para edad por grupo')
plt.ylabel('Contribución SHAP de edad')
plt.show()

---
## 8. Conclusiones

En este notebook hemos aplicado técnicas de interpretabilidad a un modelo de riesgo crediticio:

✔️ **Modelo XGBoost**: Entrenamos un clasificador con buen rendimiento (AUC > 0.7).

✔️ **Explicaciones globales con SHAP**:
- Summary plot muestra las variables más importantes y su dirección.
- Dependence plots revelan relaciones no lineales.

✔️ **Explicaciones locales**:
- Waterfall plots para casos aprobado y rechazado.
- Identificamos los factores que más contribuyeron a cada decisión.

✔️ **Informe al cliente**:
- Tradujimos las contribuciones SHAP a lenguaje comprensible.
- Cumplimos con requisitos regulatorios de explicabilidad.

**Lección clave**: SHAP permite abrir la caja negra de modelos complejos, generando confianza y cumpliendo con normativas de transparencia.

---
**Fin del Notebook de Ejercicios - Semana 13**